# When the Leaderboard Flips — M24 Capstone
## Final GitHub / UC Berkeley analysis notebook (Blocks A+B)

**Run All:** In a cloned GitHub repository, execute from the repository root or notebooks folder. In Google Colab, upload the complete `M24_GITHUB_FINAL_20261009.zip` when prompted. This notebook does not call any LLM API or require API credentials.

The notebook uses the frozen 2,000-row analytical grid and generates the updated response-length figure; the other source-controlled figures are included in `figures/`.


In [ ]:
import io, zipfile, pathlib, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from pathlib import Path
for directory in ('figures','data'):Path(directory).mkdir(exist_ok=True)
paths=[Path('data/M24_AB_2000_ANALYTICAL_GRID.csv'),Path('../data/M24_AB_2000_ANALYTICAL_GRID.csv')]
source=next((p for p in paths if p.exists()),None)
if source is None:
 zips=sorted(Path('.').glob('M24_UCB_GITHUB_*.zip'))
 if not zips:
  from google.colab import files
  print('Upload the complete M24_UCB_GITHUB_AUDITED_20261009.zip')
  uploaded=files.upload();zips=[Path(k) for k in uploaded if k.endswith('.zip')]
 assert zips,'Upload the full submission ZIP, not the figures-only ZIP.'
 with zipfile.ZipFile(zips[0]) as z:
  d=pd.read_csv(io.BytesIO(z.read('data/M24_AB_2000_ANALYTICAL_GRID.csv')),keep_default_na=False)
else:d=pd.read_csv(source,keep_default_na=False)
for k in ['end_to_end_success','acquired','parsed','response_length_words','prompt_length_words']:
 d[k]=pd.to_numeric(d[k],errors='coerce').fillna(0)
print('Frozen A+B grid:',len(d),'rows; distinct questions:',d.question_id.nunique())


## 1. Data integrity, missingness, and leakage safeguards

In [ ]:
assert len(d)==2000
assert set(d.block)=={'A','B'}
assert d.groupby(['block','model_tag']).size().eq(250).all()
assert d.groupby(['block','model_tag']).observation_id.nunique().eq(250).all()
assert d.groupby(['block','model_tag','condition']).size().eq(50).all()
assert d.groupby(['block','model_tag','benchmark']).size().eq(125).all()
print('Integrity: PASS')
display(pd.crosstab([d.block,d.model_tag],d.unresolved_reason))
assert (d.unresolved_reason=='NOT_ACQUIRED').sum()==3
assert (d.unresolved_reason=='NO_PARSEABLE_FINAL_ANSWER').sum()==2
assert d.end_to_end_success.isin([0,1]).all()

## 2. EDA: benchmarks, perturbations, response lengths and completion

In [ ]:
display(d.groupby(['block','model_tag']).agg(n=('observation_id','size'),correct=('end_to_end_success','sum'),accuracy=('end_to_end_success','mean')))
display(d.groupby(['benchmark','condition','model_tag']).end_to_end_success.agg(['size','mean']))
fig,ax=plt.subplots(figsize=(10,4));d.groupby(['model_tag','condition']).end_to_end_success.mean().unstack().plot.bar(ax=ax);ax.set_ylim(0,1.05);ax.set_ylabel('End-to-end accuracy');plt.tight_layout();plt.show()

In [ ]:
print('All descriptive figures are generated below in the canonical figure cell.')

**Figure provenance.** The response-length visualization is reproduced from the same frozen A+B analytical grid with a 25-word short-range histogram and 100-word long-tail histogram. The two panels have different bin widths and must not be compared by raw bar height. Zero-length responses are retained; acquisition and parsing status must be interpreted from the explicit quality fields. No observation is discarded or inferred.


## 3. Leaderboard inversion: ties and paired question transitions

In [ ]:
order=['baseline','paraphrase','formatting','instruction_position','irrelevant_context']
print('Ranking trajectories generated in canonical figure cell.')

## 3a. Canonical publication figures — regenerated from the frozen A+B grid

All saved figures are generated directly by this notebook. The two ranking trajectories use the preferred Colab line-plot representation. Figures are saved as PNG and SVG.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
FIG_DIR=Path('figures');FIG_DIR.mkdir(exist_ok=True)
MODELS=['Gemini','Qwen','GPT-OSS-20B','GPT-OSS-120B']
COLORS={'Gemini':'#164B6B','Qwen':'#A94D16','GPT-OSS-20B':'#176A58','GPT-OSS-120B':'#583875'}
CONDS=['baseline','paraphrase','formatting','instruction_position','irrelevant_context']
D=d.copy();D['end_to_end_success']=pd.to_numeric(D['end_to_end_success']).astype(int)
for col in ['acquired','parsed','response_length_words','prompt_length_words']:
 D[col]=pd.to_numeric(D[col],errors='coerce').fillna(0)
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'savefig.bbox':'tight','figure.dpi':120})
FIG_NAMES=[]
def save(name,fig):
 fig.savefig(FIG_DIR/(name+'.png'),dpi=210,facecolor='white')
 fig.savefig(FIG_DIR/(name+'.svg'),facecolor='white')
 plt.close(fig);FIG_NAMES.append(name)
def bar_model(tab,name,ylabel='Accuracy (%)',ylim=None):
 fig,ax=plt.subplots(figsize=(8.2,4.5));xx=np.arange(len(tab.index));wid=.18
 for j,m in enumerate(MODELS):
  if m in tab.columns:ax.bar(xx+(j-1.5)*wid,tab[m].values,width=wid,label=m,color=COLORS[m],edgecolor='white')
 ax.set_xticks(xx,tab.index,rotation=12);ax.set_ylabel(ylabel)
 if ylim:ax.set_ylim(*ylim)
 ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True);ax.legend(ncol=2,fontsize=8)
 fig.tight_layout();save(name,fig)
for b in ['A','B']:
 tab=(D[D.block==b].groupby(['condition','model_tag']).end_to_end_success.mean().unstack().reindex(CONDS)*100)
 bar_model(tab,'Fig_Block'+b+'_Accuracy')
fig,ax=plt.subplots(figsize=(8,4.3))
acc=D.groupby(['block','model_tag']).end_to_end_success.mean().unstack().loc[['A','B']]*100
xx=np.arange(2)
for j,m in enumerate(MODELS):ax.bar(xx+(j-1.5)*.19,acc[m],width=.19,color=COLORS[m],label=m)
ax.set_xticks(xx,['Block A','Block B']);ax.set_ylabel('End-to-end accuracy (%)');ax.set_ylim(0,105);ax.legend(ncol=2,fontsize=8);ax.grid(axis='y',alpha=.2);fig.tight_layout();save('Fig_Block_Comparison',fig)
bench=(D.groupby(['benchmark','model_tag']).end_to_end_success.mean().unstack()*100)
bar_model(bench,'Fig_Benchmark_Comparison')
fig,ax=plt.subplots(figsize=(8.2,4.2));statuses=D.groupby('model_tag')[['acquired','parsed','end_to_end_success']].sum().reindex(MODELS)
for j,k in enumerate(statuses.columns):ax.bar(np.arange(4)+(j-1)*.23,statuses[k],width=.23,label=k.replace('_',' '))
ax.set_xticks(range(4),MODELS);ax.set_ylabel('Observations / 500 intended');ax.legend();fig.tight_layout();save('Fig_Completeness',fig)
# preserve Colab-style line trajectories, not histograms, computed from same data
for b in ['A','B']:
 tab=D[D.block==b].groupby(['condition','model_tag']).end_to_end_success.mean().unstack().reindex(CONDS)*100
 fig,ax=plt.subplots(figsize=(9.2,4.8))
 for m in MODELS:ax.plot(range(5),tab[m].values,marker='o',lw=2.0,label=m,color=COLORS[m])
 ax.set_xticks(range(5),CONDS,rotation=14,ha='right');ax.set_ylabel('End-to-end accuracy (%)');ax.grid(axis='y',alpha=.22)
 ax.legend(ncol=2,fontsize=9);fig.tight_layout();save('Fig_Leaderboard_Trajectories_'+b,fig)
for b in ['A','B']:
 for bench in ['All','ARC_Challenge','GSM8K']:
  sub=D[D.block==b]
  if bench!='All':sub=sub[sub.benchmark.str.replace('-','_',regex=False).str.replace(' ','_',regex=False).eq(bench)]
  tab=sub.groupby(['model_tag','condition']).end_to_end_success.mean().unstack().reindex(index=MODELS,columns=CONDS)*100
  fig,ax=plt.subplots(figsize=(8.5,3.9));im=ax.imshow(tab.to_numpy(),vmin=0,vmax=100,cmap='YlGnBu',aspect='auto')
  ax.set_yticks(range(4),MODELS);ax.set_xticks(range(5),CONDS,rotation=20,ha='right')
  for i in range(4):
   for j in range(5):
    v=tab.iloc[i,j];ax.text(j,i,'—' if pd.isna(v) else f'{v:.0f}',ha='center',va='center',fontsize=9,color='white' if v>72 else 'black')
  fig.colorbar(im,ax=ax,label='Accuracy (%)');fig.tight_layout();save('Fig_Heatmap_'+b+'_'+bench,fig)
# Paired baseline effects; denominator: question-paired model x block
q=D.pivot_table(index=['block','model_tag','question_id'],columns='condition',values='end_to_end_success',aggfunc='first').reindex(columns=CONDS)
deltas=q[CONDS[1:]].sub(q['baseline'],axis=0).groupby(level='model_tag').mean().reindex(MODELS)*100
bar_model(deltas.T,'Fig_Paired_Baseline_Deltas',ylabel='Paired change (percentage points)')
trans=[]
for c in CONDS[1:]:
 t=q[['baseline',c]].dropna();trans.append([int(((t.baseline==1)&(t[c]==1)).sum()),int(((t.baseline==1)&(t[c]==0)).sum()),int(((t.baseline==0)&(t[c]==1)).sum()),int(((t.baseline==0)&(t[c]==0)).sum())])
fig,ax=plt.subplots(figsize=(8,4.5));arr=np.array(trans);bottom=np.zeros(4)
for j,label in enumerate(['Correct → correct','Correct → incorrect','Incorrect → correct','Incorrect → incorrect']):
 ax.bar(np.arange(4),arr[:,j],bottom=bottom,label=label);bottom+=arr[:,j]
ax.set_xticks(range(4),CONDS[1:],rotation=15,ha='right');ax.set_ylabel('Paired observations');ax.legend(fontsize=8);fig.tight_layout();save('Fig_Paired_Transitions',fig)
# Question-level paired bootstrap, reproducible seed
rng=np.random.default_rng(2409);ci=[]
for m in MODELS:
 s=q.xs(m,level='model_tag');base=s['baseline'].to_numpy()
 for c in CONDS[1:]:
  dif=(s[c].to_numpy()-base);dif=dif[np.isfinite(dif)]
  bs=np.mean(dif[rng.integers(0,len(dif),size=(1000,len(dif)))],axis=1)*100
  ci.append((m,c,float(np.mean(dif)*100),float(np.quantile(bs,.025)),float(np.quantile(bs,.975))))
CI=pd.DataFrame(ci,columns=['model','condition','delta_pp','ci_low_pp','ci_high_pp']);CI.to_csv('data/M24_Paired_Bootstrap_CI.csv',index=False)
fig,ax=plt.subplots(figsize=(9.2,5));yy=np.arange(len(CI))
for m in MODELS:
 sub=CI[CI.model==m];inds=sub.index.to_numpy();ax.errorbar(sub.delta_pp,inds,xerr=[sub.delta_pp-sub.ci_low_pp,sub.ci_high_pp-sub.delta_pp],fmt='o',capsize=2,color=COLORS[m],label=m)
ax.axvline(0,color='#555',ls='--',lw=1);ax.set_yticks(yy,[f'{r.model}: {r.condition}' for r in CI.itertuples()],fontsize=7);ax.set_xlabel('Paired accuracy change (percentage points)');ax.legend(ncol=2,fontsize=8);fig.tight_layout();save('Fig_Paired_Bootstrap_CI',fig)
# response length: revised two panels with disjoint bin edges
fig,(a1,a2)=plt.subplots(1,2,figsize=(12.6,5.1),gridspec_kw={'width_ratios':[1.35,1]})
hatches=['///','\\\\','xx','..'];maxlen=int(D.response_length_words.max());tailend=max(1000,int(np.ceil(maxlen/100)*100))
for m,h in zip(MODELS,hatches):
 vals=D[D.model_tag==m].response_length_words.to_numpy()
 for ax,edges in [(a1,np.arange(0,251,25)),(a2,np.arange(250,tailend+101,100))]:
  counts,_=np.histogram(vals,bins=edges);ax.stairs(counts,edges,color=COLORS[m],linewidth=1.7)
  ax.stairs(counts,edges,color=COLORS[m],linewidth=.6,fill=True,hatch=h,alpha=.12)
for ax in [a1,a2]:ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
a1.set_xlim(0,250);a2.set_xlim(250,tailend)
a1.set_xlabel('Response length (words)');a2.set_xlabel('Response length (words)');a1.set_ylabel('Responses per 25-word bin');a2.set_ylabel('Responses per 100-word bin')
a1.set_title('(a) Short responses',loc='left');a2.set_title('(b) Long-response tail',loc='left')
fig.legend([Patch(facecolor=COLORS[m],hatch=h,edgecolor=COLORS[m]) for m,h in zip(MODELS,hatches)],MODELS,ncol=4,loc='upper center',bbox_to_anchor=(.5,1.02),frameon=False)
fig.tight_layout(rect=[0,0,1,.91]);save('Fig_Response_Length',fig)
# predictive-model and importance plots are generated in subsequent ML cell
print('Descriptive figures generated:',len(FIG_NAMES))


In [ ]:
q=d.pivot_table(index=['block','model_tag','question_id'],columns='condition',values='end_to_end_success',aggfunc='first').reset_index()
rows=[]
for (b,m),g in q.groupby(['block','model_tag']):
    for c in order[1:]:
        v=g[['baseline',c]].dropna()
        rows.append({'block':b,'model':m,'condition':c,'loss':int(((v.baseline==1)&(v[c]==0)).sum()),'recovery':int(((v.baseline==0)&(v[c]==1)).sum()),'delta_pp':100*(v[c]-v.baseline).mean()})
paired=pd.DataFrame(rows);display(paired)
paired.groupby('model')[['loss','recovery']].sum().plot.bar(figsize=(9,4),title='Paired baseline transitions');plt.tight_layout();plt.show()

## 4. Group-aware supervised learning: baseline, logistic regression, random forest

In [ ]:
X=d[['model_tag','benchmark','condition','block','prompt_length_words']].copy();y=d.end_to_end_success.astype(int)
# All perturbations, models and benchmarks for the same question stay in the same split.
groups=d.question_id.astype(str)
tr,te=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=42).split(X,y,groups))
assert set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te]))
print('Train/test:',len(tr),len(te),'| Train/test question groups:',groups.iloc[tr].nunique(),groups.iloc[te].nunique())
prep=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['model_tag','benchmark','condition','block']),('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),['prompt_length_words'])])
cv=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42)
configs={'Dummy majority':(DummyClassifier(strategy='most_frequent'),{}),'Logistic regression':(LogisticRegression(max_iter=2500,class_weight='balanced'),{'model__C':[.1,1,10]}),'Random forest':(RandomForestClassifier(random_state=42,class_weight='balanced',n_jobs=-1),{'model__n_estimators':[150,300],'model__max_depth':[5,None],'model__min_samples_leaf':[2,5]})}
results=[];fitted={}
for name,(est,grid) in configs.items():
    pipe=Pipeline([('prep',prep),('model',est)])
    if grid:
        search=GridSearchCV(pipe,grid,cv=cv.split(X.iloc[tr],y.iloc[tr],groups.iloc[tr]),scoring='f1',n_jobs=-1,refit=True)
        search.fit(X.iloc[tr],y.iloc[tr]);pipe=search.best_estimator_;params=search.best_params_
    else:pipe.fit(X.iloc[tr],y.iloc[tr]);params={}
    pred=pipe.predict(X.iloc[te]);prob=pipe.predict_proba(X.iloc[te])[:,1]
    results.append({'model':name,'accuracy':accuracy_score(y.iloc[te],pred),'balanced_accuracy':balanced_accuracy_score(y.iloc[te],pred),'f1':f1_score(y.iloc[te],pred,zero_division=0),'roc_auc':roc_auc_score(y.iloc[te],prob),'best_params':str(params)})
    fitted[name]=pipe
res=pd.DataFrame(results);display(res)
res.set_index('model')[['accuracy','balanced_accuracy','roc_auc']].plot.bar(figsize=(9,4));plt.ylim(0,1.05);plt.tight_layout();plt.show()

## 5. Model interpretation and honest limitations

In [ ]:
from sklearn.inspection import permutation_importance
rf=fitted['Random forest']
pi=permutation_importance(rf,X.iloc[te],y.iloc[te],n_repeats=8,random_state=42,scoring='balanced_accuracy',n_jobs=-1)
importance=pd.DataFrame({'feature':X.columns,'mean':pi.importances_mean,'std':pi.importances_std}).sort_values('mean')
display(importance)
# Permutation-repeat standard deviations, NOT confidence intervals.
fig,ax=plt.subplots(figsize=(8.5,4.5))
ax.barh(importance.feature,importance['mean'],xerr=importance['std'],
        color='#164B6B',edgecolor='#0C2B3E',alpha=.93,
        error_kw={'ecolor':'#A34B1C','elinewidth':1.7,'capsize':4,'capthick':1.5})
ax.axvline(0,color='#333333',lw=1,ls='--')
ax.set_xlabel('Held-out permutation importance (balanced accuracy; mean ± 1 SD)')
ax.grid(axis='x',alpha=.15);ax.set_axisbelow(True)
fig.tight_layout()
save('Fig_Feature_Importance',fig)
print('Outcome prevalence:',round(y.mean(),4),'| Majority-class baseline must be considered.')
print('Permutation error bars are standard deviations across 8 repeats, not 95% confidence intervals.')
print('This is a diagnostic predictive model; no causal perturbation effects are inferred.')

## 6. Conclusions and next steps

- Report rankings separately by benchmark, block, and perturbation, and disclose ties.
- Distinguish three Qwen acquisition failures from two Gemini incomplete generations.
- Supervised correctness prediction is difficult in a high-accuracy, imbalanced dataset; do not equate overall accuracy with minority-error detection.
- Blocks C/D and repeated sampling are reserved for the journal study; the M24 deliverable is a transparent A+B interim evaluation.
- This notebook is self-contained once the complete ZIP is uploaded and requires **no API calls**.

## 7. Export and verify the 19 regenerated figures

In [ ]:
# Regenerate the two supervised-learning diagnostic figures from fitted models
import pandas as pd, numpy as np, matplotlib.pyplot as plt
if 'results' in globals():
 print('Supervised model results:',results)
if 'fitted' in globals():
 labels=[];scores=[]
 for name,est in fitted.items():
  try:
   pred=est.predict(X.iloc[te]);scores.append(float(accuracy_score(y.iloc[te],pred))*100);labels.append(name)
  except Exception as exc:print('Metric plot:',name,str(exc))
 if labels:
  fig,ax=plt.subplots(figsize=(7.6,4.2));ax.bar(labels,scores,color=['#164B6B','#176A58','#583875'][:len(labels)]);ax.set_ylabel('Held-out accuracy (%)');ax.set_ylim(0,105);fig.tight_layout();save('Fig_Model_Metrics',fig)
# Feature importance with ±1 SD error bars was already saved in Section 5.
assert 'Fig_Feature_Importance' in FIG_NAMES
print('Generated figure count:',len(FIG_NAMES))
print('Generated:',sorted(FIG_NAMES))
assert len(FIG_NAMES)==19, f'Expected 19 figures, found {len(FIG_NAMES)}'
import zipfile
with zipfile.ZipFile('M24_Colab_Regenerated_Figures.zip','w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(FIG_DIR.glob('*')):z.write(p,p.as_posix())
print('Download M24_Colab_Regenerated_Figures.zip from Colab Files to update your local repository.')


## 8. Visual audit: all 19 figures displayed inline
This gallery displays each PNG produced by the notebook. It is a visual check, not an additional analysis.


In [ ]:
# Visible verification gallery: display every generated figure in Colab
from IPython.display import display, Image, Markdown
from pathlib import Path
visible_figures = sorted(Path('figures').glob('Fig_*.png'))
print(f'VISIBLE FIGURE AUDIT: {len(visible_figures)} PNG figures')
assert len(visible_figures) == 19, f'Expected 19 PNG figures, found {len(visible_figures)}'
for figure_path in visible_figures:
    display(Markdown(f'### {figure_path.stem}'))
    display(Image(filename=str(figure_path), width=850))
print('All 19 figures displayed above. Download M24_Colab_Regenerated_Figures.zip from Files.')
